# EIA Testing

The purpose of this notebook is to test the ingestion process from EIA API  

In [22]:
import requests

import pandas as pd
from datetime import datetime
import duckdb
import pyarrow as pa

import os
from dotenv import load_dotenv

from sqlalchemy import (
    Column, 
    Table, 
    MetaData, 
    text, 
    Text, 
    Numeric, 
    Integer, 
    BigInteger, 
    Boolean, 
    DateTime, 
    create_engine, 
    inspect
)

from sqlalchemy.dialects.postgresql import insert as pg_insert

import json

In [2]:
TYPE_MAP = {
    'TEXT' : Text,
    'NUMERIC' : Numeric,
    'INTEGER' : Integer,
    'BIGINT' : BigInteger,
    'BOOLEAN' : Boolean,
    'TIMESTAMP' : DateTime,
}

In [3]:
load_dotenv()

API_KEY = os.getenv("EIA_API_KEY")

# Consider adding a try-except block to handle potential errors in the API request and missing API keys

In [4]:
BASE_URL = "https://api.eia.gov/v2"

In [25]:
def eia_fetch_all(route: str, params: dict) -> duckdb.DuckDBPyRelation:
    """
    Paginated fetch - handles datasets larger than 5000 rows.

    Parameters:
    - route: The API endpoint route (e.g., 'series')
    - params: A dictionary of parameters to include in the API request

    Returns:
    - A DuckDB relation of all pages combined
    """

    all_records = []
    offset = 0
    page_size = 5000 # max per request from eia.gov
    
    while True: 

        params['offset'] = offset
        params['length'] = page_size
        params['api_key'] = API_KEY

        url = f"{BASE_URL}/{route}/data/"
    
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()  # Raise an error for bad status codes
            
            payload = response.json()['response']      
            records = payload['data']
            total = int(payload.get("total", 0))

            all_records.extend(records)
            offset += page_size

            print(f"[INFO] Fetched {min(offset, total)}/{total} records from {route}")

            if offset >= total:
                break  # Exit the loop if we've fetched all records
            
        except requests.exceptions.RequestException as e:
            print(f"An error occurred while fetching data: {e}")
            break # Return nothing
    
    if (not all_records):
        return duckdb.sql("SELECT 1 WHERE FALSE")

    all_records_arrow = pa.Table.from_pylist(all_records)
    return duckdb.sql("SELECT * FROM all_records_arrow")

In [6]:
def create_postgres_engine(user: str, password: str, host: str, port: int, database: str) -> create_engine:
    """
    Create a SQLAlchemy engine for connecting to a PostgreSQL database.

    Parameters:
    - user: Database username
    - password: Database password
    - host: Database host (e.g., 'localhost')
    - port: Database port (e.g., 5432)
    - database: Database name

    Returns:
    - A SQLAlchemy engine instance
    """
    connection_string = f"postgresql://{user}:{password}@{host}:{port}/{database}"
    engine = create_engine(connection_string)
    return engine

In [7]:
def create_schema_if_not_exists(engine: create_engine, schema_name: str):
    """
    Create a schema in the PostgreSQL database if it does not already exist.

    Parameters:
    - engine: A SQLAlchemy engine instance
    - schema_name: The name of the schema to create
    """
    with engine.begin() as connection:
        connection.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema_name};"))
    print(f"Schema '{schema_name}' is ready.")

In [36]:
def write_to_postgres(
        relation: duckdb.DuckDBPyRelation, 
        table_name: str, 
        schema: dict, 
        engine: create_engine,
        pg_schema: str = "bronze"
        ) -> None: 
    """
    Dynamically create or upsert a PostgreSQL table from a DuckDB relation.

    Schema format:
        { "column_name": ("DATA_TYPE", is_primary_key) }

    Behaviour:
        - Table doesn't exist  → creates it with defined PKs, then inserts
        - Table exists         → upserts on conflict of PK columns
    """

    # --- Auto-create schema ---
    create_schema_if_not_exists(engine, pg_schema)

    metadata = MetaData(schema=pg_schema)
    inspector = inspect(engine)

    # --- Build Column objects from schema dict --- 
    columns = []
    pk_columns = []

    for col_name, (dtype, is_pk) in schema.items():
        col_type = TYPE_MAP.get(dtype.upper())
        if not col_type:
            raise ValueError(f"Unsupported data type '{dtype}' for column '{col_name}'")
        
        columns.append(Column(col_name, col_type, primary_key=is_pk))
        if is_pk:
            pk_columns.append(col_name)
        
    if not pk_columns:
        raise ValueError("At least one primary key column must be defined in the schema.")

    # --- Define table object ---
    table = Table(table_name, metadata, *columns)

    # --- Create table if it doesn't exist --- 
    existing_tables = inspector.get_table_names(schema=pg_schema)
    if table_name not in existing_tables:
        metadata.create_all(engine)
        print(f"[INFO] Created '{pg_schema}.{table_name}' with PKs: {pk_columns}")

    # --- Extract records from DuckDB relation --- 
    # Only keep columns that exist in both the schema and the relation
    available_cols = relation.columns
    selected_cols = [c for c in schema.keys() if c in available_cols]

    if (not selected_cols):
        raise ValueError("No overlapping columns between schema and relation.")

    # Project down to only the columns we need, then pull as list of dicts
    col_list = ", ".join(f'"{c}"' for c in selected_cols)
    records = (
        duckdb.sql(f"SELECT {col_list} FROM relation")
        .fetchdf()
        .to_dict(orient="records")
    )

    if (not records):
        print(f"[WARN] No records to write to '{pg_schema}.{table_name}'.")
        return

    # --- Upsert --- 
    non_pk_cols = [c for c in selected_cols if c not in pk_columns]

    with engine.begin() as conn:
        stmt = pg_insert(table).values(records)

        if non_pk_cols:
            upsert_stmt = stmt.on_conflict_do_update(
                index_elements=pk_columns,
                set_={col: stmt.excluded[col] for col in non_pk_cols}
            )
        else:
            upsert_stmt = stmt.on_conflict_do_nothing(index_elements=pk_columns)
        
        conn.execute(upsert_stmt)
    
    print(f"[INFO] Upserted {len(records)} records into '{pg_schema}.{table_name}' on PK conflict of {pk_columns}.")




Execute

In [9]:
# Create the engine for PostgreSQL connection
engine = create_postgres_engine(
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=int(os.getenv("POSTGRES_PORT", 5432)),
    database=os.getenv("POSTGRES_DB")
)

In [17]:
# Analyze metadata for a specific dataset to understand its structure and available filters

def eia_metadata(route: str) -> dict:
    url = f"{BASE_URL}/{route}/"
    r = requests.get(url, params={"api_key": API_KEY})
    return r.json()["response"]

# See all available frequencies, facets, and data columns for production
meta = eia_metadata("petroleum/crd/crpdn")
print(meta["frequency"])   # shows available frequencies
print(meta["facets"])      # shows filterable dimensions
print(meta["data"])        # shows available value columns

[{'id': 'monthly', 'description': 'One data point for each month.', 'query': 'M', 'format': 'YYYY-MM'}, {'id': 'annual', 'description': 'One data point for each calendar year.', 'query': 'A', 'format': 'YYYY'}]
[{'id': 'duoarea', 'description': 'DuoArea'}, {'id': 'product', 'description': 'Product'}, {'id': 'process', 'description': 'Process'}, {'id': 'series', 'description': 'Series'}]
{'value': []}


### What's Available?

In [11]:
def eia_get_routes(route: str = "") -> pd.DataFrame:
    """Walk the EIA API route tree to discover child routes."""
    url = f"{BASE_URL}/{route}" if route else BASE_URL
    r = requests.get(url, params={"api_key": API_KEY}, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["response"].get("routes", []))

# Start at petroleum and walk down
print("=== PETROLEUM ROUTES ===")
print(eia_get_routes("petroleum"))          # see all petroleum sub-routes
print(eia_get_routes("petroleum/sum"))      # supply summary sub-routes
print(eia_get_routes("natural-gas/prod"))    # natural gas production sub-routes

=== PETROLEUM ROUTES ===
     id                           name
0   sum                        Summary
1   pri                         Prices
2   crd  Crude Reserves and Production
3   pnp        Refining and Processing
4  move  Imports/Exports and Movements
5  stoc                         Stocks
6  cons              Consumption/Sales
       id                                               name  \
0    b100  U.S. Biodiesel Production, Capacity, Sales, an...   
1     snd                             Supply and Disposition   
2    sndw                            Weekly Supply Estimates   
3  crdsnd                U.S. Crude Oil Supply & Disposition   
4     mkt            Prices, Sales Volumes & Stocks by State   

                                         description  
0  EIA discontinued updates data after December 2...  
1  Imports at the PAD District level represent th...  
2  Domestic crude oil production includes lease c...  
3  Adjustments include an adjustment for crude oi...  
4  

In [12]:
def eia_get_facet_values(route: str, facet: str) -> pd.DataFrame:
    """
    Query the EIA API facet endpoint to get all valid values for a given facet.
    This is the authoritative way to discover valid codes before building params.
    """
    url = f"{BASE_URL}/{route}/facet/{facet}/"
    r = requests.get(url, params={"api_key": API_KEY}, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["response"]["facets"])

# Get every valid duoarea code for the production route
df_areas = eia_get_facet_values("petroleum/crd/crpdn", "duoarea")
print(df_areas)

# Check petroleum product codes as well
df_products = eia_get_facet_values("petroleum/crd/crpdn", "product")
print(df_products)

# Check what's available under the NGL/petroleum supply route
df_ngl_products = eia_get_facet_values("petroleum/sum/sndw", "product")
print(df_ngl_products)

# Or check the natural gas plant liquids processing route
df_ng_products = eia_get_facet_values("natural-gas/prod/sum", "product")
print(df_ng_products)

      id        name
0    SNE      USA-NE
1    SNY    NEW YORK
2    SOH        OHIO
3    R20      PADD 2
4    SID      USA-ID
5    SIL      USA-IL
6    SCA  CALIFORNIA
7    SND      USA-ND
8    SNM      USA-NM
9    SNV      USA-NV
10   SUT      USA-UT
11   SWY      USA-WY
12  RAKS          NA
13   SKS      USA-KS
14   SKY      USA-KY
15   R30      PADD 3
16   SSD      USA-SD
17   SVA      USA-VA
18   SAL      USA-AL
19   SAR      USA-AR
20   SFL     FLORIDA
21   SMS      USA-MS
22   R10      PADD 1
23   SWV      USA-WV
24   SMI      USA-MI
25   SAK      USA-AK
26  R3FM          NA
27   SCO    COLORADO
28   SMT      USA-MT
29   SOK      USA-OK
30   R50      PADD 5
31   STN      USA-TN
32   R5F          NA
33   R40      PADD 4
34   NUS        U.S.
35   SAZ      USA-AZ
36   SIN      USA-IN
37   SLA      USA-LA
38   SMO      USA-MO
39   SPA      USA-PA
40   STX       TEXAS
       id           name
0    EPC0      Crude Oil
1  EPCANS  ANS Crude Oil
           id                              

### Natural Gas Liquids (NGLs) && Crude Oil Production

In [44]:
products = [
    "EPC0",
    "EPLLPZ"
]

duoareas = [
    "S-TX",
    "S-NM",
    "S-ND",
    "S-CO",
    "S-WY",
    # CHECK
    "S-OK",
    "S-LA"
]

In [41]:
# --- Crude Oil Production (monthly, by state) ---
params = {
    "frequency": "monthly",
    "data[0]": "value",
    "facets[series][]": "MCRFPTX2",   # Texas field production of crude oil
    "start": "2015-01",
    "end": "2024-12",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
production_rel = eia_fetch_all("petroleum/crd/crpdn", params)


[INFO] Fetched 120/120 records from petroleum/crd/crpdn


In [32]:
print(production_rel.describe())

┌─────────┬─────────┬─────────┬───────────┬─────────┬──────────────┬─────────┬──────────────────┬──────────┬────────────────────────────────────────────────────────────────┬─────────┬─────────┐
│  aggr   │ period  │ duoarea │ area-name │ product │ product-name │ process │   process-name   │  series  │                       series-description                       │  value  │  units  │
│ varchar │ varchar │ varchar │  varchar  │ varchar │   varchar    │ varchar │     varchar      │ varchar  │                            varchar                             │ varchar │ varchar │
├─────────┼─────────┼─────────┼───────────┼─────────┼──────────────┼─────────┼──────────────────┼──────────┼────────────────────────────────────────────────────────────────┼─────────┼─────────┤
│ count   │ 120     │ 120     │ 120       │ 120     │ 120          │ 120     │ 120              │ 120      │ 120                                                            │ 120     │ 120     │
│ mean    │ NULL    │ NULL    

In [42]:
production_rel = duckdb.sql("""
        SELECT
            CAST(period  || '-01' AS TIMESTAMP)   AS period
            ,duoarea
            ,"area-name"                AS "area_name"
            ,product
            ,"product-name"             AS "product_name"
            ,process         
            ,"process-name"             AS "process_name"
            ,series
            ,"series-description"       AS "series_description"
            ,TRY_CAST(value AS DOUBLE)  AS value
            ,units
        FROM production_rel
    """)

In [43]:
# Define the schema for the PostgreSQL table
write_to_postgres(
    relation=production_rel,
    table_name="crude_oil_production",
    schema={
        "period":               ("TIMESTAMP", True),
        "duoarea":              ("TEXT", False),
        "area_name":            ("TEXT", True),
        "product":              ("TEXT", True),
        "product_name":         ("TEXT", False),
        "process":              ("TEXT", True),
        "process_name":         ("TEXT", False),
        "series":               ("TEXT", True),
        "series_description":   ("TEXT", False),
        "value":                ("NUMERIC", False),
        "units":                ("TEXT", False)
    },
    engine=engine,
)

Schema 'bronze' is ready.
[INFO] Upserted 120 records into 'bronze.crude_oil_production' on PK conflict of ['period', 'area_name', 'product', 'process', 'series'].


### WTI Crude Spot Price (daily)

In [ ]:
# --- WTI Crude Spot Price (daily) ---
params = {
    "frequency": "daily",
    "data[0]": "value",
    "facets[series][]": "RWTC",
    "start": "2015-01-01",
    "end": "2024-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_wti = eia_fetch("petroleum/pri/spt", params)
display(df_wti)

### Natural Gas Spot Price (Henry Hub, daily)

In [ ]:
# --- Natural Gas Spot Price (Henry Hub, daily) ---
params = {
    "frequency": "daily",
    "data[0]": "value",
    "facets[series][]": "RNGWHHD",
    "start": "2015-01-01",
    "end": "2024-12-31",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_ng_price = eia_fetch("natural-gas/pri/sum", params)
display(df_ng_price)

### Weekly Crude Inventory (US Total Stocks)

In [ ]:
# --- Weekly Crude Inventory (US total stocks) ---
params = {
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WTTSTUS1",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
}
df_inventory = eia_fetch("petroleum/stoc/wstk", params)
display(df_inventory)